[LC 125](https://leetcode.com/problems/valid-palindrome/description/)

# My Solution
Point to note
- replaceAll
- regex negation [^
- isEmpty() = str.length() == 0 (jdk 6+)

In [12]:
class Solution {
    public static boolean isPalindrome(String s) {
       // if(s.isEmpty() return true, as per eg 3) 
       // isEmpty() (jdk6+) just checks the length, not whitespaces
       if(s.length() == 0) return true;
        String regex = "[^a-zA-Z0-9]";
        s = s.replaceAll(regex,"");
        s = s.toLowerCase();
        int i = 0;
        int j = s.length() - 1;

        while(i < j){
            if(s.charAt(i)==s.charAt(j)){
                i++;
                j--;
            }else{
                return false;
            }
        }
        return true;
    }
}

# Optimal 1

Instead of mutating the string and creating copies, professional solutions use two pointers that move inward from both ends, checking characters on the fly while skipping non-alphanumeric characters using 
- `Character.isLetterOrDigit()` and
- `Character.toLowerCase()`

In [13]:
class Solution1 {
    public boolean isPalindrome(String s) {
        int i = 0;
        int j = s.length() - 1;
        
        while (i < j) {
            char start = s.charAt(i);
            char end = s.charAt(j);
            
            // If the character at 'i' is not a letter or digit, skip it
            if (!Character.isLetterOrDigit(start)) {
                i++;
            } 
            // If the character at 'j' is not a letter or digit, skip it
            else if (!Character.isLetterOrDigit(end)) {
                j--;
            } 
            // Both are alphanumeric, compare them case-insensitively
            else {
                if (Character.toLowerCase(start) != Character.toLowerCase(end)) {
                    return false;
                }
                i++;
                j--;
            }
        }
        return true;
    }
}

# Recursive

In [ ]:
class Solution {
public boolean isPalindrome(String s) {
s = s.replaceAll("[^a-zA-Z0-9]", "");
return isPalindromeCheck(s.toLowerCase(),0,s.length()-1);
}

private boolean isPalindromeCheck(String s,int left,int right){
if(left >= right){
return true;
}
if(s.charAt(left) != s.charAt(right)){
return false;
}

return isPalindromeCheck(s,left+1,right-1);
}
} 


In [5]:
public class Main{
    public static void main(String[] args){
        String s = "A man, a plan, a canal: Panama";
        System.out.println(Solution.isPalindrome(s));
        String s1 = "race a car";
        System.out.println(Solution.isPalindrome(s1));
        String s2 = " ";
        System.out.println(Solution.isPalindrome(s2));

    }
}

In [6]:
Main.main(new String[]{});

true
false
true


# Recursion Tradeoff

Pros: It's very readable, compact, and showcases a solid understanding of recursion.

Cons (Performance): Because it uses recursion, it adds overhead to the call stack. For a string of length $N$, it uses $O(N)$ stack space in the worst case, plus the memory already allocated by replaceAll() and toLowerCase().

**Yes, recursion definitely increases latency** (and memory usage) in Java, which is why recursion solution took 17 ms while yours ran in a lightning-fast 1 ms.

Several factors explain why recursion slows things down:

* **Stack Frame Overhead:** Every time a recursive function calls itself, Java must create a new "stack frame" in memory to keep track of local variables, parameters, and where to return when the function finishes. Doing this thousands of times adds massive overhead.
* **CPU Cycle Cost:** Jumping back and forth between stack frames requires extra CPU operations compared to staying inside a single loop.
* **The `StackOverflowError` Risk:** The constraints for this LeetCode problem allow strings up to $200,000$ characters long. A recursion depth that deep will actually **crash** with a `StackOverflowError` on large test cases in Java because the JVM runs out of stack memory.

Your iterative two-pointer solution avoids all of that overhead by reusing the exact same stack frame and utilizing simple CPU register operations. That's why iteration wins big here!

# Metaspace vs Stack
Metaspace. = Host OS memory, so stackoverlfow and outofheap memory hadly comes into picture, isn't that correct ?

That is a common point of confusion, but **Metaspace** and the **JVM Stack** are entirely different memory regions!

Here is how JVM memory is actually partitioned and why `StackOverflowError` still happens regardless of how much OS memory you have:

### 1. The Core JVM Memory Regions

* **Metaspace:** This is where the JVM stores class definitions, method metadata, and static variables. Since Java 8, it dynamically grows using host OS memory, meaning you rarely hit "out of metaspace" unless you have a massive class-loading leak.
* **The JVM Stack (Thread Stack):** Every individual thread in Java gets its **own private stack** (controlled by the `-Xss` flag, typically around 1 MB per thread). This stack stores local primitive variables, method arguments, and **stack frames** for active method calls.
* **The Heap:** Where dynamic objects (`String`, arrays, `ArrayList`, etc.) live.

### 2. Why `StackOverflowError` Happens

When a recursive function calls itself, a new stack frame is pushed onto that specific thread's **JVM Stack**.

Because each thread's stack size is fixed (usually 1 MB), if your recursion goes too deep (e.g., 10,000 recursive calls deep for a long string), it will exhaust that 1 MB limit and throw a `StackOverflowError` immediately—even if your host OS has 64 GB of free RAM. The OS memory cannot dynamically expand a single thread's assigned stack region past its configured limit.

### 3. What about OutOfMemoryError (Heap)?

If you instead created a massive data structure or kept allocating new strings inside that recursion, you would eventually hit a `java.lang.OutOfMemoryError: Java heap space`. But deep recursion specifically kills you on **stack depth**, not heap size or metaspace.